# Recommendation Bot — Combined Trading Strategy

Multi-indicator strategy combining four signals into daily trade recommendations.

| # | Indicator | Source file | Role | Output |
|---|-----------|-------------|------|--------|
| 1 | Permutation Entropy | `Permutation_Entropy.ipynb` | Chaos / order filter | 0 / 1 |
| 2 | MACD | `MACD.ipynb` | Trend direction | -1 / 0 / 1 |
| 3 | Fibonacci Retracement | `Fibonacci_Retracement.ipynb` | Key support / resistance | -1 / 0 / 1 |
| 4 | MA Crossover | `Permutation_Entropy.ipynb` | Trend confirmation | -1 / 0 / 1 |

All indicator functions live in **`indicators.py`** and are imported here — no re-implementation.

**Recommendation logic:**
- PE = 0 (chaotic) → **hold**
- PE = 1 (orderly) + 3-of-3 → **strong\_buy / strong\_sell**
- PE = 1 (orderly) + 2-of-3 → **buy / sell**
- mixed → **hold**

In [ ]:
import os
import csv
import math
import time
import warnings
from datetime import datetime

import numpy as np
import matplotlib.pyplot as plt
import requests
from scipy import stats

warnings.filterwarnings('ignore')

In [ ]:
# ---- Import all indicator functions from the shared module ------------------
#
#   indicators.py  consolidates every indicator used across the project:
#     indicator_1_pe           <- Permutation_Entropy.ipynb  cell c07
#     indicator_2_macd         <- MACD.ipynb                 cell 2
#     indicator_5_fibonacci    <- Fibonacci_Retracement.ipynb cell 2
#     indicator_4_ma_crossover <- Permutation_Entropy.ipynb  cell c10
#
from indicators import (
    indicator_1_pe,
    indicator_2_macd,
    indicator_5_fibonacci,
    indicator_4_ma_crossover,
)

print('Indicators loaded from indicators.py')
print('  indicator_1_pe           (Permutation_Entropy.ipynb)')
print('  indicator_2_macd         (MACD.ipynb)')
print('  indicator_5_fibonacci    (Fibonacci_Retracement.ipynb)')
print('  indicator_4_ma_crossover (Permutation_Entropy.ipynb)')

In [ ]:
# ---- Configuration ----------------------------------------------------------
API_KEY    = 'ZKMMTO1ATDBLXH2K'
TICKERS    = ['AAPL', 'MSFT', 'JPM']
INDEX      = 'SPY'
START_DATE = '2022-01-01'
END_DATE   = '2024-12-31'
RISK_FREE  = 0.05          # annual risk-free rate

TRAIN_DAYS = 252            # walk-forward: in-sample window  (~1 year)
TEST_DAYS  = 126            # walk-forward: out-of-sample window (~6 months)

In [ ]:
def _download_prices(ticker, start_date, end_date, api_key):
    """Download daily adjusted close prices from AlphaVantage.
    Returns list of [ticker, date, adjusted_close, log_return].
    """
    url = ('https://www.alphavantage.co/query'
           '?function=TIME_SERIES_DAILY_ADJUSTED'
           '&symbol=' + ticker +
           '&outputsize=full'
           '&apikey=' + api_key)
    response = requests.get(url)
    data     = response.json()

    ts = data.get('Time Series (Daily)', {})
    if not ts:
        print('  Warning: no data for ' + ticker + '. Keys: ' + str(list(data.keys())))
        return []

    raw = []
    for date_str, vals in ts.items():
        if start_date <= date_str <= end_date:
            raw.append([ticker, date_str, float(vals['5. adjusted close'])])
    raw.sort(key=lambda x: x[1])

    result = []
    for i, rec in enumerate(raw):
        lr = 0.0 if i == 0 else math.log(rec[2] / raw[i - 1][2])
        result.append([rec[0], rec[1], rec[2], lr])
    return result


def download_all_data(tickers, index, start_date, end_date, api_key):
    """Download + cache all ticker data. One CSV per ticker.
    Skips download if CSV already exists (respects AlphaVantage rate limits).

    Returns
    -------
    dict  {ticker: [[ticker, date, price, log_return], ...]}
    """
    all_data = {}
    for ticker in tickers + [index]:
        csv_file = ticker + '_data.csv'
        if os.path.exists(csv_file):
            print('  ' + ticker + ': loading from ' + csv_file)
            records = []
            with open(csv_file, newline='') as f:
                reader = csv.reader(f)
                next(reader)
                for row in reader:
                    records.append([row[0], row[1], float(row[2]), float(row[3])])
        else:
            print('  ' + ticker + ': downloading from AlphaVantage...')
            records = _download_prices(ticker, start_date, end_date, api_key)
            with open(csv_file, 'w', newline='') as f:
                writer = csv.writer(f)
                writer.writerow(['ticker', 'date', 'adjusted_close', 'log_return'])
                writer.writerows(records)
            print('    Saved ' + str(len(records)) + ' rows to ' + csv_file)
            time.sleep(15)  # AlphaVantage free tier: 5 req/min
        all_data[ticker] = records
    return all_data

In [ ]:
# Download on first run; subsequent runs load from cached CSVs
data = download_all_data(TICKERS, INDEX, START_DATE, END_DATE, API_KEY)
for ticker, records in data.items():
    print(ticker + ': ' + str(len(records)) + ' records  ('
          + records[0][1] + ' to ' + records[-1][1] + ')')

---
## Strategy

PE acts as the gate — if the market is chaotic (PE = 0), no trade is placed.
When PE = 1 (orderly), a majority vote of the three directional indicators determines the recommendation.

| Indicator | Bull (+1) | Bear (-1) |
|-----------|-----------|-----------|
| MACD | MACD line > signal line | MACD line < signal line |
| Fibonacci | Near support in uptrend | Near resistance in downtrend |
| MA Crossover | 20-day SMA > 50-day SMA | 20-day SMA < 50-day SMA |

In [ ]:
def get_recommendation(pe_sig, macd_sig, fib_sig, ma_sig):
    """Combine four indicator signals into a 5-level recommendation.

    PE = 0  -> 'hold'  (chaotic regime: step aside)
    PE = 1  -> majority vote on {MACD, Fibonacci, MA Crossover}:
        3 bullish -> 'strong_buy'   |  3 bearish -> 'strong_sell'
        2 bullish -> 'buy'          |  2 bearish -> 'sell'
        else      -> 'hold'
    """
    if pe_sig == 0:
        return 'hold'

    bullish = sum(1 for s in [macd_sig, fib_sig, ma_sig] if s ==  1)
    bearish = sum(1 for s in [macd_sig, fib_sig, ma_sig] if s == -1)

    if   bullish == 3: return 'strong_buy'
    elif bullish == 2: return 'buy'
    elif bearish == 3: return 'strong_sell'
    elif bearish == 2: return 'sell'
    return 'hold'


# Smoke test
print(get_recommendation(1,  1,  1,  1))   # all bullish  -> strong_buy
print(get_recommendation(1, -1, -1,  1))   # 2 bearish    -> sell
print(get_recommendation(1,  1, -1,  0))   # mixed        -> hold
print(get_recommendation(0,  1,  1,  1))   # PE=0 chaotic -> hold

In [ ]:
def execute_trade(recommendation, log_return_next_day):
    """Execute a single trade based on a recommendation (README item 8).

    Returns (log_return, direction) or (None, None) for 'hold'.
    """
    if recommendation in ('strong_buy', 'buy'):
        return log_return_next_day, 'long'
    elif recommendation in ('strong_sell', 'sell'):
        return -log_return_next_day, 'short'
    return None, None


def execute_trades(data_dict, tickers, index_ticker='SPY'):
    """Run the full indicator pipeline for each ticker (README item 9).

    Holds a position until the direction reverses or signal goes flat.
    Stores one log-return per closed trade.

    Returns
    -------
    trade_log_returns        : list[float]
    trade_dates              : list[tuple]  (entry_date, exit_date, ticker, direction)
    market_returns_at_trades : list[float]  market return on each exit day
    """
    trade_log_returns        = []
    trade_dates              = []
    market_returns_at_trades = []
    spy_ret = {r[1]: r[3] for r in data_dict.get(index_ticker, [])}

    WARMUP = 60   # max warmup: Fibonacci=60, MA=50, MACD=35, PE=32

    for ticker in tickers:
        rows   = data_dict[ticker]
        prices = [r[2] for r in rows]
        dates  = [r[1] for r in rows]

        position    = 0
        entry_price = None
        entry_date  = None

        for i in range(WARMUP, len(prices)):
            pe  = indicator_1_pe(prices, i)
            mac = indicator_2_macd(prices, i)
            fib = indicator_5_fibonacci(prices, i)
            ma  = indicator_4_ma_crossover(prices, i)
            rec = get_recommendation(pe, mac, fib, ma)

            desired = (1  if rec in ('strong_buy',  'buy')
                  else -1 if rec in ('strong_sell', 'sell')
                  else 0)

            if position != 0 and desired != position:
                log_ret = math.log(prices[i] / entry_price)
                if position == -1:
                    log_ret = -log_ret
                trade_log_returns.append(log_ret)
                direction = 'long' if position == 1 else 'short'
                trade_dates.append((entry_date, dates[i], ticker, direction))
                market_returns_at_trades.append(spy_ret.get(dates[i], 0.0))
                position    = 0
                entry_price = None

            if position == 0 and desired != 0:
                position    = desired
                entry_price = prices[i]
                entry_date  = dates[i]

    return trade_log_returns, trade_dates, market_returns_at_trades

In [ ]:
def compute_performance(trade_log_returns, trade_dates, market_returns_at_trades,
                        rf_annual=RISK_FREE, label='Strategy'):
    """Compute and print all required performance metrics (README item 10).

    a. Number of trades per month
    b. Average return + t-test for statistical significance
    c. Average return for long trades
    d. Average return for short trades
    e. Cumulative return, annualised
    f. Sharpe Ratio, annualised
    g. Sortino Ratio, annualised
    h. Jensen's Alpha (beta from market returns on trade close dates)
    i. VaR at the 5% level
    """
    if not trade_log_returns:
        print('No trades executed.')
        return {}

    rets    = np.array(trade_log_returns)
    mkt     = np.array(market_returns_at_trades)
    n       = len(rets)
    long_r  = [r for r, (_, _, _, d) in zip(trade_log_returns, trade_dates) if d == 'long']
    short_r = [r for r, (_, _, _, d) in zip(trade_log_returns, trade_dates) if d == 'short']

    months = {}
    for _, exit_d, _, _ in trade_dates:
        months[exit_d[:7]] = months.get(exit_d[:7], 0) + 1
    avg_tpm = n / max(len(months), 1)

    mean_ret      = float(np.mean(rets))
    t_stat, p_val = stats.ttest_1samp(rets, 0)
    avg_long  = float(np.mean(long_r))  if long_r  else float('nan')
    avg_short = float(np.mean(short_r)) if short_r else float('nan')

    cum_ret = float(np.sum(rets))
    d0      = datetime.strptime(trade_dates[0][0],  '%Y-%m-%d')
    d1      = datetime.strptime(trade_dates[-1][1], '%Y-%m-%d')
    years   = max((d1 - d0).days / 365.25, 1e-6)
    ann_ret = cum_ret / years

    rf_d   = rf_annual / 252
    excess = rets - rf_d
    sharpe = float(np.mean(excess) / np.std(excess, ddof=1) * np.sqrt(252))              if np.std(excess) > 0 else 0.0

    down    = rets[rets < 0]
    d_std   = float(np.std(down, ddof=1)) if len(down) > 1 else 1e-9
    sortino = float((mean_ret / d_std) * np.sqrt(252))

    if len(mkt) > 1 and np.var(mkt) > 0:
        cov   = np.cov(rets, mkt)
        beta  = cov[0, 1] / np.var(mkt, ddof=1)
        alpha = (mean_ret - (rf_d + beta * (np.mean(mkt) - rf_d))) * 252
    else:
        beta  = float('nan')
        alpha = float('nan')

    var5 = float(np.percentile(rets, 5))

    SEP = '=' * 56
    sig = ('***' if p_val < 0.01 else '**' if p_val < 0.05 else '*' if p_val < 0.10 else 'n.s.')
    print(SEP)
    print('  PERFORMANCE REPORT -- ' + label)
    print(SEP)
    print('Total Trades           : ' + str(n) +
          '  (Long: ' + str(len(long_r)) + ', Short: ' + str(len(short_r)) + ')')
    print('')
    print('a. Avg Trades / Month  : ' + str(round(avg_tpm, 2)))
    print('')
    print('b. Mean Log-Return     : ' + str(round(mean_ret, 6)))
    print('   t-stat              : ' + str(round(float(t_stat), 4)))
    print('   p-value             : ' + str(round(float(p_val), 4)) + '  (' + sig + ')')
    print('')
    print('c. Avg Return (Longs)  : ' + str(round(avg_long,  6)))
    print('d. Avg Return (Shorts) : ' + str(round(avg_short, 6)))
    print('')
    print('e. Cumulative Return   : ' + str(round(cum_ret * 100, 2)) + '%')
    print('   Annualised Return   : ' + str(round(ann_ret * 100, 2)) + '%')
    print('')
    print('f. Sharpe Ratio        : ' + str(round(sharpe,  4)))
    print('')
    print('g. Sortino Ratio       : ' + str(round(sortino, 4)))
    print('')
    print('h. Beta                : ' + str(round(float(beta),  4)))
    print('   Jensens Alpha       : ' + str(round(float(alpha), 6)))
    print('')
    print('i. VaR (5%)            : ' + str(round(var5 * 100, 2)) + '%')
    print(SEP)

    return dict(n_trades=n, avg_tpm=avg_tpm, mean_return=mean_ret,
                t_stat=float(t_stat), p_value=float(p_val),
                avg_long=avg_long, avg_short=avg_short,
                cum_return=cum_ret, ann_return=ann_ret,
                sharpe=sharpe, sortino=sortino,
                beta=float(beta), jensens_alpha=float(alpha), var5=var5)

In [ ]:
def walk_forward_backtest(data_dict, tickers, index_ticker='SPY',
                          train_days=TRAIN_DAYS, test_days=TEST_DAYS):
    """Walk-forward out-of-sample validation.
    No parameter re-fitting: tests stability with fixed hyperparameters.
    """
    print('=' * 60)
    print('           WALK-FORWARD BACKTEST RESULTS')
    print('  Train: ' + str(train_days) + ' days  |  Test: ' + str(test_days) + ' days')
    print('-' * 60)

    n_dates = min(len(v) for v in data_dict.values())
    folds   = []
    start   = train_days
    while start + test_days <= n_dates:
        folds.append((start, start + test_days))
        start += test_days

    print('  Total folds: ' + str(len(folds)))
    print('-' * 60)
    print('  Fold   Test Start     Test End    Ann.Ret%   Sharpe   Win%')
    print('-' * 60)

    all_oos = []
    for k, (ts, te) in enumerate(folds):
        fold_data = {t: data_dict[t][ts:te] for t in tickers + [index_ticker]}
        tr, td, mr = execute_trades(fold_data, tickers, index_ticker)
        if not tr:
            print('   ' + str(k + 1) + '   (no trades in this fold)')
            continue

        all_oos.extend(tr)
        r   = np.array(tr)
        d0  = datetime.strptime(td[0][0],  '%Y-%m-%d')
        d1  = datetime.strptime(td[-1][1], '%Y-%m-%d')
        yrs = max((d1 - d0).days / 365.25, 1e-6)
        ann = float(np.sum(r)) / yrs
        exc = r - RISK_FREE / 252
        sh  = (float(np.mean(exc) / np.std(exc, ddof=1) * np.sqrt(252))
               if np.std(exc) > 0 else 0.0)
        win = float(np.mean(r > 0)) * 100
        s   = fold_data[tickers[0]][0][1]  if fold_data[tickers[0]] else '---'
        e   = fold_data[tickers[0]][-1][1] if fold_data[tickers[0]] else '---'
        line = ('  ' + str(k + 1).rjust(4) + '   ' + s + '   ' + e +
                '   ' + str(round(ann * 100, 2)).rjust(8) + '%' +
                '   ' + str(round(sh, 3)).rjust(6) +
                '   ' + str(round(win, 1)).rjust(5) + '%')
        print(line)

    if all_oos:
        r_all  = np.array(all_oos)
        t, p   = stats.ttest_1samp(r_all, 0)
        print('-' * 60)
        print('  OOS avg return : ' + str(round(float(np.mean(r_all)), 6)) +
              '   t=' + str(round(float(t), 3)) +
              '   p=' + str(round(float(p), 4)))
    print('=' * 60)
    return all_oos if all_oos else []

---
## Results

Run the cells below to execute the strategy and view all performance metrics.

In [ ]:
# ---- Full-sample backtest: all tickers combined ----------------------------
trade_returns, trade_info, mkt_rets = execute_trades(data, TICKERS, INDEX)
perf = compute_performance(trade_returns, trade_info, mkt_rets,
                           label='All Tickers Combined')

In [ ]:
# ---- Per-ticker performance breakdown --------------------------------------
for ticker in TICKERS:
    tr, td, mr = execute_trades(
        {ticker: data[ticker], INDEX: data[INDEX]}, [ticker], INDEX
    )
    if tr:
        compute_performance(tr, td, mr, label=ticker)
    else:
        print('[' + ticker + ']  No trades generated.')

In [ ]:
# ---- Walk-forward out-of-sample validation ---------------------------------
oos_returns = walk_forward_backtest(data, TICKERS, INDEX)

In [ ]:
# ---- Equity curve ----------------------------------------------------------
if trade_returns:
    cum_curve  = np.cumsum(trade_returns)
    exit_dates = [td[1] for td in trade_info]
    dt_exits   = [datetime.strptime(d, '%Y-%m-%d') for d in exit_dates]

    fig, ax = plt.subplots(figsize=(13, 5))
    ax.plot(dt_exits, cum_curve, linewidth=1.5, color='steelblue',
            label='Strategy cumulative log-return')
    ax.axhline(0, color='black', linewidth=0.6, linestyle='--')
    ax.set_title('Cumulative Log-Return -- PE + MACD + Fibonacci + MA Crossover',
                 fontsize=13)
    ax.set_xlabel('Trade Exit Date')
    ax.set_ylabel('Cumulative Log-Return')
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig('recommendation_bot_equity_curve.png', dpi=150)
    plt.show()
    print('Equity curve saved to recommendation_bot_equity_curve.png')
else:
    print('No trades to plot.')